
# Exploratory Data Analysis (EDA)

A complete step-by-step guide to **Exploratory Data Analysis** — what it is, why we do it, and how to perform
Univariate, Bivariate and Multivariate analysis on numerical and categorical data — followed by hands-on
practice on the **Titanic** dataset and then a full EDA + Feature Engineering exercise on an **insurance** dataset.

---

## Setup

We first import the libraries we will use for the entire notebook.

| Library | Purpose |
|---------|---------|
| `pandas` | Loading & manipulating tabular data (`DataFrame`) |
| `numpy` | Fast numerical operations |
| `seaborn` | High-level statistical visualisation |
| `matplotlib.pyplot` | Low-level plotting control |

```python
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
```

In [ ]:

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Make plots look a little nicer
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)



We load the **Titanic** training dataset. Make sure `train.csv` is in the same folder as this notebook
(or change the path below).

```python
df = pd.read_csv('train.csv')
df.head()
```

In [ ]:

df = pd.read_csv('train.csv')

df.head()



## Why do EDA?

EDA is the **pre-modelling** step that helps us:

- **Model building** — understanding the data before choosing/creating models.
- **Analysis and reporting** — discovering and communicating patterns.
- **Validate assumptions** — checking whether statistical assumptions hold.
- **Handling missing values** — spotting gaps and deciding how to fill/exclude them.
- **Feature engineering** — creating new, more informative columns from existing ones.
- **Detecting outliers** — finding extreme values that may distort a model.

> **Remember:** EDA is an **iterative process**. You keep going back and forth between exploring
> and cleaning — there is no "one pass and you are done".



## Column Types

Before analysing we classify every column:

- **Numerical** — continuous/discrete numbers you can do arithmetic on: `Age`, `Fare`, `PassengerId`.
- **Categorical** — values that are labels/groups: `Survived`, `Pclass`, `Sex`, `SibSp`, `Parch`, `Embarked`.
- **Mixed** — messy text that needs parsing: `Name`, `Ticket`, `Cabin`.

Let's inspect the raw data types:

```python
df.info()
```

In [ ]:

df.info()



## Univariate Analysis

**Univariate analysis** focuses on analysing **each feature independently**.

### Two main goals of univariate analysis:
1. **Distribution analysis** — the shape, central tendency and dispersion of each feature.
2. **Identifying potential issues** — outliers, skewness and missing values.

---

### The shape of a data distribution

The **shape** refers to the overall pattern of the data when plotted. Common shapes:

- **Normal Distribution** — symmetrical, bell-shaped; mean = median = mode; most data in the middle,
  frequencies taper off towards the tails.
- **Skewed Distribution** — not symmetrical, one tail longer than the other.
  - **Positively (right) skewed**: long tail on the right (mean > median).
  - **Negatively (left) skewed**: long tail on the left (mean < median).
- **Bimodal Distribution** — two peaks (two modes).
- **Uniform Distribution** — every value equally likely.

> Knowing the shape matters because it reveals outliers, skewness, and tells us which statistical tests
> and models are appropriate.

---

### Dispersion (spread)

**Dispersion** measures how far the values spread out from the central tendency. Common measures:

- **Range** — largest − smallest value.
- **Variance** — average of the squared deviations from the mean.
- **Standard Deviation** — square root of variance; spread in the *same units* as the data.
- **Interquartile Range (IQR)** — value between the 25th percentile (Q1) and 75th percentile (Q3).

Dispersion helps identify extreme values and outliers.



## Steps of Univariate Analysis on **Numerical** columns

1. **Descriptive Statistics** — compute mean, median, mode, std, range, quartiles. Gives a general sense of
   centre and spread and reveals skewness/outliers.
2. **Visualizations** — histograms, box plots and density (KDE) plots to *see* the distribution.
3. **Identifying Outliers** — via box plots / IQR. Decide if they are measurement errors, entry errors or
   legitimate values, and whether to keep/drop them.
4. **Skewness** — check and decide whether to transform the data or use robust methods.
5. **Conclusion** — document findings and decide how to proceed.

---

### Age

```python
df['Age'].describe()
df['Age'].plot(kind='hist', bins=20)
df['Age'].plot(kind='kde')
df['Age'].skew()
df['Age'].plot(kind='box')
```

**Conclusions (Age):**
- Age is **almost** normally distributed.
- **~20%** of values are missing.
- There are some **outliers** (a few very old passengers).


In [ ]:

df['Age'].describe()


In [ ]:

df['Age'].plot(kind='hist', bins=20)
plt.title('Distribution of Age')
plt.show()


In [ ]:

df['Age'].plot(kind='kde')
plt.title('Age - Density Plot (KDE)')
plt.show()


In [ ]:

print('Skewness of Age:', df['Age'].skew())


In [ ]:

df['Age'].plot(kind='box')
plt.title('Age - Box Plot')
plt.show()


In [ ]:

# Look at the extreme (old) ages
df[df['Age'] > 65]


In [ ]:

# Proportion of missing Age values
df['Age'].isnull().sum() / len(df['Age'])



### Fare

```python
df['Fare'].describe()
df['Fare'].plot(kind='hist')
df['Fare'].plot(kind='kde')
df['Fare'].skew()
df['Fare'].plot(kind='box')
df[df['Fare'] > 250]
df['Fare'].isnull().sum()
```

**Conclusions (Fare):**
- The data is **highly positively skewed**.
- `Fare` actually contains the **group fare** (a whole family/ticket), not the *individual* fare.
  This might be an issue — we may need to create a new column called **individual fare**.


In [ ]:

df['Fare'].describe()


In [ ]:

df['Fare'].plot(kind='hist', bins=50)
plt.title('Distribution of Fare')
plt.show()


In [ ]:

df['Fare'].plot(kind='kde')
plt.title('Fare - Density Plot')
plt.show()


In [ ]:

print('Skewness of Fare:', df['Fare'].skew())


In [ ]:

df['Fare'].plot(kind='box')
plt.title('Fare - Box Plot')
plt.show()


In [ ]:

# Extreme fares
df[df['Fare'] > 250]


In [ ]:

print('Missing Fare values:', df['Fare'].isnull().sum())



## Steps of Univariate Analysis on **Categorical** columns

1. **Descriptive Statistics** — frequency distribution of each category and relative frequencies.
2. **Visualizations** — count plots and pie charts to *see* the distribution and spot anomalies.
3. **Missing Values** — check and decide how to handle (impute or exclude).
4. **Conclusion** — document findings and decide how to proceed.

### Survived, Sex, Embarked

```python
df['Embarked'].value_counts()
df['Embarked'].value_counts().plot(kind='bar')
df['Embarked'].value_counts().plot(kind='pie', autopct='%0.1f%%')
df['Sex'].isnull().sum()
```

**Conclusions:**
- `Parch` and `SibSp` can be merged to form a new column **family_size**.
- We can also create a new column **is_alone**.


In [ ]:

df['Embarked'].value_counts()


In [ ]:

df['Embarked'].value_counts().plot(kind='bar')
plt.title('Embarked - Count Plot')
plt.show()


In [ ]:

df['Embarked'].value_counts().plot(kind='pie', autopct='%0.1f%%')
plt.title('Embarked - Pie Chart')
plt.ylabel('')
plt.show()


In [ ]:

print('Missing Sex values:', df['Sex'].isnull().sum())


In [ ]:

# Also check value counts of the other categorical columns
for col in ['Sex', 'Pclass', 'SibSp', 'Parch']:
    print(f'--- {col} ---')
    print(df[col].value_counts())
    print()



## Bivariate Analysis

**Bivariate analysis** looks at **two columns together** to understand their relationship.

### Steps
1. **Select 2 columns.**
2. **Understand the type of relationship** between them:
    1. **Numerical – Numerical**
        - Scatter plots (and regression plots), 2D histograms, 2D KDE plots.
        - Check the **correlation coefficient** for a linear relationship.
    2. **Numerical – Categorical**
        - Compare the distribution of the numerical variable across categories.
        - bar plots, box plots, KDE plots, violin plots, even scatter plots.
    3. **Categorical – Categorical**
        - **Cross-tabulations / contingency tables** (`pd.crosstab`) showing how one categorical column
          splits by another.
        - heatmaps, stacked bar plots, treemaps.
3. **Write your conclusions** — always document what you observe.

---

### Categorical – Categorical: Survival vs other categories

```python
sns.heatmap(pd.crosstab(df['Survived'], df['Pclass'], normalize='columns') * 100)
pd.crosstab(df['Survived'], df['Sex'], normalize='columns') * 100
pd.crosstab(df['Survived'], df['Embarked'], normalize='columns') * 100
pd.crosstab(df['Sex'], df['Embarked'], normalize='columns') * 100
pd.crosstab(df['Pclass'], df['Embarked'], normalize='columns') * 100
```

> `normalize='columns'` computes, for **each** category of the second variable, what percentage were
> Survived (1) vs Not Survived (0). `* 100` turns proportions into percentages.


In [ ]:

# Survival vs Passenger Class
sns.heatmap(pd.crosstab(df['Survived'], df['Pclass'], normalize='columns') * 100,
            annot=True, fmt='.1f', cmap='coolwarm')
plt.title('Survival % by Passenger Class (columns normalized)')
plt.show()


In [ ]:

# Survival vs Sex
pd.crosstab(df['Survived'], df['Sex'], normalize='columns') * 100


In [ ]:

# Survival vs Embarked
pd.crosstab(df['Survived'], df['Embarked'], normalize='columns') * 100


In [ ]:

# Sex vs Embarked
pd.crosstab(df['Sex'], df['Embarked'], normalize='columns') * 100


In [ ]:

# Pclass vs Embarked
pd.crosstab(df['Pclass'], df['Embarked'], normalize='columns') * 100



### Numerical – Categorical: Survival vs Age

```python
df[df['Survived'] == 1]['Age'].plot(kind='kde', label='Survived')
df[df['Survived'] == 0]['Age'].plot(kind='kde', label='Not Survived')
plt.legend()
plt.show()
```

**Conclusion:** The age distributions of survivors and non-survivors largely overlap; age alone
is not hugely predictive, but children and older adults differ slightly in survival rates.


In [ ]:

df[df['Survived'] == 1]['Age'].plot(kind='kde', label='Survived')
df[df['Survived'] == 0]['Age'].plot(kind='kde', label='Not Survived')
plt.legend()
plt.title('Age distribution: Survived vs Not Survived')
plt.show()


In [ ]:

# Mean age of 1st class passengers (just an example of numerical-categorical summary)
print(df[df['Pclass'] == 1]['Age'].mean())



## Feature Engineering on the Titanic data

Feature engineering turns raw columns into **more informative** ones. We will create:

1. **`individual_fare`** — `Fare` is the *group* fare summed across a family/ticket; dividing by
   family size gives the per-person fare.
2. **`family_size`** — `SibSp + Parch + 1` (the passenger themself).
3. **`family_type`** — alone / small / large based on family size.
4. **`surname`** — from `Name`.
5. **`title`** — the person's title (`Mr.`, `Mrs.`, `Master.`, `Miss.`, ...).
6. **`deck`** — first letter of `Cabin`, with missing values filled as `'M'`.

```python
df['SibSp'].value_counts()
df[df['Ticket'] == 'CA. 2343']
df[df['Name'].str.contains('Sage')]
```

> Note: we also load the **test** set and concatenate, so we can inspect passengers with the same ticket.


In [ ]:

df['SibSp'].value_counts()


In [ ]:

df[df['Ticket'] == 'CA. 2343']


In [ ]:

df[df['Name'].str.contains('Sage')]


In [ ]:

# Load test set and combine so ticket groups can be found across both files
df1 = pd.read_csv('test.csv')
df = pd.concat([df, df1])
print('Combined shape:', df.shape)


In [ ]:

df[df['Ticket'] == 'CA 2144']



### 1. Individual Fare

```python
df['individual_fare'] = df['Fare'] / (df['SibSp'] + df['Parch'] + 1)
```

This divides the group fare by the total people sharing the ticket. Compare `individual_fare` vs `Fare`.


In [ ]:

df['individual_fare'] = df['Fare'] / (df['SibSp'] + df['Parch'] + 1)

df['individual_fare'].plot(kind='box')
plt.title('Individual Fare - Box Plot')
plt.show()


In [ ]:

df[['individual_fare', 'Fare']].describe()



### 2. Family Size and 3. Family Type

```python
df['family_size'] = df['SibSp'] + df['Parch'] + 1
```

- `1`        -> `alone`
- `2 - 4`    -> `small`
- `> 5`      -> `large`

```python
def transform_family_size(num):
    if num == 1:
        return 'alone'
    elif num > 1 and num < 5:
        return 'small'
    else:
        return 'large'

df['family_type'] = df['family_size'].apply(transform_family_size)
```


In [ ]:

df['family_size'] = df['SibSp'] + df['Parch'] + 1

def transform_family_size(num):
    if num == 1:
        return 'alone'
    elif num > 1 and num < 5:
        return 'small'
    else:
        return 'large'

df['family_type'] = df['family_size'].apply(transform_family_size)

df.head()


In [ ]:

# Survival rate by family type
pd.crosstab(df['Survived'], df['family_type'], normalize='columns') * 100



### 4. Surname and 5. Title

```python
df['surname'] = df['Name'].str.split(',').str.get(0)
df['title']   = df['Name'].str.split(',').str.get(1).str.strip().str.split(' ').str.get(0)
```


In [ ]:

df['surname'] = df['Name'].str.split(',').str.get(0)
df['title'] = df['Name'].str.split(',').str.get(1).str.strip().str.split(' ').str.get(0)

df.head()


In [ ]:

# Survival by title, restricting to the main titles
temp_df = df[df['title'].isin(['Mr.', 'Miss.', 'Mrs.', 'Master.'])]
pd.crosstab(temp_df['Survived'], temp_df['title'], normalize='columns') * 100



**Conclusion:** `Master.` (boys) and `Mrs.`/`Miss.` (women/young girls) survived at much higher rates
than `Mr.` — title is a strong signal (because of the "women and children first" policy).

### Fold rare titles into `other`

```python
df['title'] = df['title'].str.replace('Rev.', 'other')
df['title'] = df['title'].str.replace('Dr.',  'other')
df['title'] = df['title'].str.replace('Col.', 'other')
df['title'] = df['title'].str.replace('Major.','other')
df['title'] = df['title'].str.replace('Capt.','other')
df['title'] = df['title'].str.replace('the',  'other')
df['title'] = df['title'].str.replace('Jonkheer.','other')
```


In [ ]:

df['title'] = df['title'].str.replace('Rev.', 'other')
df['title'] = df['title'].str.replace('Dr.', 'other')
df['title'] = df['title'].str.replace('Col.', 'other')
df['title'] = df['title'].str.replace('Major.', 'other')
df['title'] = df['title'].str.replace('Capt.', 'other')
df['title'] = df['title'].str.replace('the', 'other')
df['title'] = df['title'].str.replace('Jonkheer.', 'other')

df['title'].value_counts()



### 6. Deck from Cabin

```python
df['Cabin'].fillna('M', inplace=True)
df['deck'] = df['Cabin'].str[0]
```

We fill missing cabins with `'M'` (missing) and take the first letter as the **deck**.


In [ ]:

print('Missing Cabin: %.1f%%' % (df['Cabin'].isnull().sum() / len(df['Cabin']) * 100))
df['Cabin'].fillna('M', inplace=True)
df['Cabin'].value_counts()


In [ ]:

df['deck'] = df['Cabin'].str[0]
df['deck'].value_counts()


In [ ]:

# Deck vs Pclass
pd.crosstab(df['deck'], df['Pclass'])


In [ ]:

# Survival by deck (normalized by row, stacked)
pd.crosstab(df['deck'], df['Survived'], normalize='index').plot(kind='bar', stacked=True)
plt.title('Survival by Deck')
plt.show()



## Correlation and Pair Plots

```python
sns.heatmap(df.corr())
sns.pairplot(df1)
```

The **correlation heatmap** shows the linear relationship (Pearson r) between every pair of numerical
columns. The **pair plot** shows every pair of variables side by side (scatter + histograms on diagonal).


In [ ]:

sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()


In [ ]:

# Pair plot on the un-engineered test data
sns.pairplot(df1)
plt.show()



---

# Hands-on Task: EDA on an Insurance Dataset

Now we apply the **full EDA process** to a real dataset: **Insurance Claim Analysis — Demographic and Health**.

- Dataset source (Kaggle): `https://www.kaggle.com/datasets/thedevastator/insurance-claim-analysis-demographic-and-health`
- This is the classic **insurance** dataset with **medical costs** (`charges`).

### The Columns (typical for this dataset):
| Column | Type | Meaning |
|--------|------|---------|
| `age` | numerical | age of the insured person |
| `sex` | categorical | male / female |
| `bmi` | numerical | body mass index |
| `children` | numerical | number of dependants covered |
| `smoker` | categorical | yes / no |
| `region` | categorical | northeast / northwest / southeast / southwest |
| `charges` | numerical | individual medical costs billed by health insurance (**target**) |

> **How to get the data:** Download `insurance.csv` from the Kaggle link (or use any copy of the
> classic `insurance.csv`) and place it in the same folder as this notebook, then run the next cell.

---

## Step 0 — Load the data

```python
ins = pd.read_csv('insurance.csv')
ins.head()
ins.info()
ins.describe(include='all')
```


In [ ]:

ins = pd.read_csv('insurance.csv')

ins.head()


In [ ]:

ins.info()


In [ ]:

ins.describe(include='all')



**Quick observations**
- We have `1338` rows, `7` columns.
- No obvious missing values (all counts equal 1338) — but we will **verify** below.
- Numeric: `age`, `bmi`, `children`, `charges`. Categorical: `sex`, `smoker`, `region`.

---

## Step 1 — Univariate Analysis

### 1.1 Numerical columns
For each numerical column we compute descriptive statistics, histograms, KDE, box plots, skewness,
and check for outliers / missing values.


In [ ]:

# Missing values check across the whole dataframe
print('Missing values per column:')
print(ins.isnull().sum())
print()
print('Duplicate rows:', ins.duplicated().sum())


In [ ]:

# Descriptive statistics for all numerical columns
ins[['age', 'bmi', 'children', 'charges']].describe()


In [ ]:

# Skewness of the numerical columns
ins[['age', 'bmi', 'children', 'charges']].skew()



#### Age
- Roughly uniform between 18–64, with a small bump near 18–20.
- No extreme outliers; `skew` is close to 0 (slightly negative).


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ins['age'].plot(kind='hist', bins=30, ax=axes[0]); axes[0].set_title('Age Histogram')
ins['age'].plot(kind='kde', ax=axes[1]); axes[1].set_title('Age KDE')
ins['age'].plot(kind='box', ax=axes[2]); axes[2].set_title('Age Box')
plt.tight_layout(); plt.show()



#### BMI
- Roughly **normal (bell-shaped)** with mean ≈ 30.6.
- A few **outliers** on the high side (bmi > 45) — worth investigating but they are realistic values.


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ins['bmi'].plot(kind='hist', bins=30, ax=axes[0]); axes[0].set_title('BMI Histogram')
ins['bmi'].plot(kind='kde', ax=axes[1]); axes[1].set_title('BMI KDE')
ins['bmi'].plot(kind='box', ax=axes[2]); axes[2].set_title('BMI Box')
plt.tight_layout(); plt.show()



#### Children
- Strongly **right-skewed**; most people have `0` children, counts drop off quickly.
- It is a **count** (discrete) variable, so a value-count view is more useful than a histogram.


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ins['children'].value_counts().sort_index().plot(kind='bar', ax=axes[0]); axes[0].set_title('Children - Count Plot')
ins['children'].plot(kind='box', ax=axes[1]); axes[1].set_title('Children Box')
plt.tight_layout(); plt.show()



#### Charges (target)
- **Heavily right-skewed** (skew ≈ 1.5). This is typical of medical-cost data — most people are cheap,
  a minority are very expensive.
- There is a distinct second cluster of high charges (≈ 30k–50k) which is strongly related to
  **smokers** (we will confirm in bivariate analysis).
- **Implication for modelling:** a log transform is frequently applied to `charges` to reduce skewness.

```python
np.log1p(ins['charges']).skew()
```


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ins['charges'].plot(kind='hist', bins=40, ax=axes[0]); axes[0].set_title('Charges Histogram')
ins['charges'].plot(kind='kde', ax=axes[1]); axes[1].set_title('Charges KDE')
ins['charges'].plot(kind='box', ax=axes[2]); axes[2].set_title('Charges Box')
plt.tight_layout(); plt.show()


In [ ]:

# Log transform reduces the skew drastically
print('Charges skew:', round(ins['charges'].skew(), 3))
print('Log(charges) skew:', round(np.log1p(ins['charges']).skew(), 3))



### 1.2 Categorical columns
For each categorical column we look at **frequency distributions** with count plots / pie charts
and check for missing values.

**Conclusions:**
- `sex` is well balanced (~50/50).
- `smoker` is imbalanced — only ~20% are smokers (important class-imbalance note).
- `region` is balanced across the four regions.


In [ ]:

for col in ['sex', 'smoker', 'region']:
    print(f'--- {col} ---')
    print(ins[col].value_counts())
    print()


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ins['sex'].value_counts().plot(kind='bar', ax=axes[0]); axes[0].set_title('Sex')
ins['smoker'].value_counts().plot(kind='bar', ax=axes[1]); axes[1].set_title('Smoker')
ins['region'].value_counts().plot(kind='bar', ax=axes[2]); axes[2].set_title('Region')
plt.tight_layout(); plt.show()



## Step 2 — Bivariate Analysis

### 2.1 Categorical → Numerical: effect on charges

We compare `charges` across the categories of `sex`, `smoker`, `region`, and `children`.
Box plots and bar plots are ideal here.

**Conclusions:**
- **Smoker is the dominant driver of charges.** Smokers pay far more (median ≈ 34k vs ≈ 7.3k for non-smokers).
  The bimodal-looking distribution of `charges` is really the two smoker groups overlapping.
- **Age and BMI** each add costs, but their effect is much smaller than smoking.
- `sex` and `region` have only marginal effects on average charges.
- **Children:** up to ~3 children slightly raises costs, then plateaus.


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.boxplot(data=ins, x='smoker', y='charges', ax=axes[0]); axes[0].set_title('Charges by Smoker')
sns.boxplot(data=ins, x='sex', y='charges', ax=axes[1]); axes[1].set_title('Charges by Sex')
sns.boxplot(data=ins, x='region', y='charges', ax=axes[2]); axes[2].set_title('Charges by Region')

plt.tight_layout(); plt.show()


In [ ]:

# Average charges by category (quick numeric summary)
print(ins.groupby('smoker')['charges'].mean().round(2))
print()
print(ins.groupby('sex')['charges'].mean().round(2))
print()
print(ins.groupby('region')['charges'].mean().round(2))


In [ ]:

# Average charges by number of children
ins.groupby('children')['charges'].mean().round(2)



### 2.2 Numerical → Numerical: correlation with charges

Scatter plots plus the **correlation coefficient** tell us the linear relationship with `charges`.

**Conclusions:**
- `age` vs `charges`: positive relation but with two visible clusters (again smokers vs non-smokers).
- `bmi` vs `charges`: weak positive correlation overall; correlation is much stronger **within smokers**.
- Correlation matrix: `charges` correlates most with `age` (≈ 0.30) and `bmi` (≈ 0.20) among numerics;
  the real driver, `smoker`, is categorical.


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=ins, x='age', y='charges', hue='smoker', ax=axes[0]); axes[0].set_title('Charges vs Age (colored by smoker)')
sns.scatterplot(data=ins, x='bmi', y='charges', hue='smoker', ax=axes[1]); axes[1].set_title('Charges vs BMI (colored by smoker)')
plt.tight_layout(); plt.show()


In [ ]:

sns.heatmap(ins[['age', 'bmi', 'children', 'charges']].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap (numerical columns)')
plt.show()



### 2.3 Categorical → Categorical

**Conclusions:**
- **Smoking is not evenly spread** — in the **southeast** the *smoker share* is slightly higher.
- `sex` and `smoker` are roughly balanced across each other (no strong interaction).


In [ ]:

pd.crosstab(ins['smoker'], ins['region'], normalize='columns') * 100


In [ ]:

pd.crosstab(ins['smoker'], ins['sex'], normalize='columns') * 100



## Step 3 — Multivariate (summary view)

A `pairplot` colored by `smoker` captures the key relationships at once: smokers form a clearly
separate, higher-cost cloud across `age` and `bmi`.


In [ ]:

sns.pairplot(ins, hue='smoker', diag_kind='kde')
plt.show()



## Step 4 — Feature Engineering

Based on the EDA we create new, more predictive columns:

### 4.1 BMI categories (`bmi_category`)
Standard WHO thresholds:
- `< 18.5`  → `underweight`
- `18.5–24.9` → `normal`
- `25–29.9` → `overweight`
- `>= 30`   → `obese`

### 4.2 Age group (`age_group`)
- `18–30` → `young`, `31–50` → `middle`, `> 50` → `senior`

### 4.3 `insurance_risk` (composite health flag)
A simple health-risk label combining smoking and BMI:
- `high risk`  → smoker AND (bmi >= 30)
- `moderate risk` → smoker OR (bmi >= 30)
- `low risk`    → otherwise (non-smoker and bmi < 30)

### 4.4 `log_charges`
A log-transformed target to reduce the heavy right-skew before modelling:

```python
ins['log_charges'] = np.log1p(ins['charges'])
```


In [ ]:

def bmi_cat(bmi):
    if bmi < 18.5:
        return 'underweight'
    elif bmi < 25:
        return 'normal'
    elif bmi < 30:
        return 'overweight'
    else:
        return 'obese'

ins['bmi_category'] = ins['bmi'].apply(bmi_cat)
ins.sample(5)


In [ ]:

def age_group(a):
    if a <= 30:
        return 'young'
    elif a <= 50:
        return 'middle'
    else:
        return 'senior'

ins['age_group'] = ins['age'].apply(age_group)
ins.sample(5)


In [ ]:

def risk(row):
    s = row['smoker'] == 'yes'
    o = row['bmi'] >= 30
    if s and o:
        return 'high risk'
    elif s or o:
        return 'moderate risk'
    else:
        return 'low risk'

ins['insurance_risk'] = ins.apply(risk, axis=1)
ins.sample(5)


In [ ]:

ins['log_charges'] = np.log1p(ins['charges'])
ins.head()



### Validate the engineered features

**Conclusions:**
- `bmi_category`/`insurance_risk` show a clean, monotonic rise in average charges — confirming they
  are useful predictors.
- `log_charges` is far less skewed than `charges` (skew ≈ 0.3 vs ≈ 1.5) — good for linear models.


In [ ]:

ins.groupby('bmi_category')['charges'].mean().round(2)


In [ ]:

ins.groupby('insurance_risk')['charges'].mean().round(2)


In [ ]:

ins.groupby('age_group')['charges'].mean().round(2)


In [ ]:

pd.crosstab(ins['insurance_risk'], ins['smoker'])


In [ ]:

print('charges skew:', round(ins['charges'].skew(), 3))
print('log_charges skew:', round(ins['log_charges'].skew(), 3))
ins['log_charges'].plot(kind='hist', bins=40)
plt.title('Log(charges) distribution - much more symmetric')
plt.show()



## Step 5 — Summary & Conclusions

**Univariate:**
- `charges` is highly **right-skewed** with two overlapping cost clusters → use `log_charges` for modelling.
- `bmi` is near-normal with a few high outliers (realistic). `age` is roughly uniform. `children` is a
  right-skewed count (`0` is most common).
- No missing values; a few **duplicate rows** can be dropped.

**Bivariate / Multivariate:**
- **`smoker` is the single strongest driver of medical charges** (median roughly 3–4× higher).
- `age` and `bmi` have weaker, positive effects that are clearer once `smoker` is controlled for.
- `sex` and `region` have only marginal average effects.

**Feature engineering performed:**
- `bmi_category`, `age_group`, `insurance_risk` (composite risk), and `log_charges`.
- The new risk features show a monotonic relationship with cost — useful for modelling.

**Next steps (if modelling):**
- One-hot / label encode `sex`, `smoker`, `region`, and the new categorical features.
- Scale `age` / `bmi`.
- Fit a regression model (e.g. Linear / Random Forest) on `log_charges`; invert with `expm1` for predictions.



## Your Turn

Try extending this notebook:

1. Drop duplicates with `ins = ins.drop_duplicates()` and re-run the stats.
2. Analyse `children` vs charges with a bar plot of *average* charges.
3. Build a simple model: split data, train a `RandomForestRegressor` on **log_charges**, and report RMSE.
4. Explore other bivariate combos (e.g. `age_group` × `smoker` vs charges) with an `hue` box plot.
